# **Quantum Computing — the Practical Way**
### *QPlayLearn*


## **Installation**

First, we install the packages we need in the current environment. They will not be installed on your local machine.


In [ ]:
# Qiskit is the open-source library for quantum computing founded by IBM
! pip install qiskit qiskit-aer qiskit-ibm-runtime
! pip install matplotlib pylatexenc


## **Importing packages**

We import all the packages we are going to need to run the code.  
Remember to run this cell before every Sandbox!


In [ ]:
import qiskit as qk
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# Packages for graphical representations and plots
import matplotlib as mpl
import matplotlib.pyplot as plt

# Math library
import numpy as np


## **SANDBOX — Bell states**


When working with several qubits, we can use the same commands introduced in *Quantum Computing — the Soft Way* to apply gates and inspect the amplitudes of the resulting state.

Let’s first see how these commands extend to a multi-qubit circuit. We will then use them to create Bell states and introduce measurement on a quantum circuit.

### **0 — Build a multi-qubit circuit**

To apply a gate in a circuit with multiple qubits, specify the index of the qubit on which it acts. For a CNOT, specify both the control and the target:

```python
qc.x(qubit)
qc.cx(control, target)
```

Try adding gates below. Then draw the circuit and inspect its statevector.


In [ ]:
# Create a four-qubit circuit
num_qubits = 4
qc = qk.QuantumCircuit(num_qubits)

# Your turn: apply an X gate to q2


# Your turn: add a CNOT with your chosen control and target
# qc.cx(..., ...)

# Draw the circuit and inspect its state
qc.draw(output="mpl")
state = Statevector(qc)
print("|psi> = ", state.data)



**A note on Qiskit’s qubit ordering**


Before interpreting the statevector, we need to understand how Qiskit orders the qubits.

In this course and most textbooks, we label a generic two-qubit state by writing $q_0$ first and $q_1$ second:

$$
|\psi\rangle
=a|0_{q_0}0_{q_1}\rangle
+b|0_{q_0}1_{q_1}\rangle
+c|1_{q_0}0_{q_1}\rangle
+d|1_{q_0}1_{q_1}\rangle.
$$

where the notation $ | \ 0_{q_0} \ 0_{q_1} \ \rangle$ indicates the computational basis state where the first qubit labelled $q_0$ is in state $| 0 \rangle$ and the second qubit $q_1$ is in state $| 0 \rangle$, and so on.

With this convention, the coefficients are listed as

$$[a,b,c,d].$$

Qiskit writes the qubits in the opposite order inside a basis state: $|q_1q_0\rangle$, an ordering called "little-endian" and inherited from computer science. The same physical state and its statevector are therefore displayed as

$$
|\psi\rangle
=a|0_{q_1}0_{q_0}\rangle
+c|0_{q_1}1_{q_0}\rangle
+b|1_{q_1}0_{q_0}\rangle
+d|1_{q_1}1_{q_0}\rangle,
$$

$$[a,c,b,d].$$

Let's see an example. Suppose we apply an $X$ gate to $q_0$, leaving $q_1$ in $|0\rangle$. Qiskit writes $q_1$ on the left and $q_0$ on the right, so the displayed state and its statevector are

$$|q_1q_0\rangle=|01\rangle,$$

$$[0,1,0,0].$$

This reversed order also appears in the bit strings returned by measurements, so let's keep it in mind!

For more information on how extend this to more than 2 qubits, check out this video: https://youtu.be/EiqHj3_Avps?feature=shared


### **1 — Creating entanglement**

With more than one qubit, we can play with entanglement. Earlier in the course, we saw that a Hadamard gate followed by a CNOT creates the maximally entangled Bell state

$$
|\Phi^+\rangle=\frac{|00\rangle+|11\rangle}{\sqrt{2}}.
$$

Although Qiskit writes the qubits as $|q_1q_0\rangle$, $|\Phi^+\rangle$ has the same form when the two positions are exchanged.


In [ ]:
# Create the Bell state |Phi+>
qc = qk.QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

qc.draw(output="mpl")


Inspect its statevector. Which computational-basis states have non-zero amplitudes?


In [ ]:
state = Statevector(qc)
print("|Phi^+> = ", state.data)

### **2 — Introducing measurement**

So far, we have inspected quantum states through their representation as vectors. The squared modulus of each amplitude gives the probability of obtaining the corresponding computational-basis state.

On a quantum device, we can extract information only through measurement, which returns classical information. If we want to estimate probabilities of finding the qubits in the states of the computational basis, a single measurement is not enough. We need instead to run the circuit and measure many times. Each repetition is called a **shot**.

For the Bell state $|\Phi^+\rangle$, the only non-zero amplitudes correspond to $|00\rangle$ and $|11\rangle$. The squared modulus of each amplitude is $1/2$, so measuring both qubits returns either `00` or `11`, each with probability $1/2$. Let's measure it.

Change `num_shots` to `1`, `10`, `100` and `1000`. How do the observed frequencies change?

In [ ]:
# Set AerSimulator as the backend on which we run the circuit
sim_bknd = AerSimulator()

# Create the Bell-state circuit again
num_qubits = 2
qc = qk.QuantumCircuit(num_qubits)
qc.h(0)
qc.cx(0, 1)

# Add a measurement to every qubit
# Qiskit automatically creates the classical bits that store the outcomes
qc.measure_all()
qc.draw(output="mpl")


# If in the future you want to measure specific qubits and not all og´f them, define first a number of classical bits large enough to store the outcomes from the measured qubits.

# For example, here we measure qubit 1 and store the outcome in the classical bit 1
#num_bits = 2
#qc = qk.QuantumCircuit(num_qubits, num_bits)
#qc.measure(1, 1)
#qc.draw(output="mpl")

In [ ]:
# Now run the circuit on the backend and get the results
# Your turn: try changing the number of shots
num_shots = 1
res = sim_bknd.run(qc, shots = num_shots).result()
counts = res.get_counts()

# Print the results 
print("Measurement outcomes\n", counts)
plot_histogram([counts],legend=['Simulator'])

With a small number of shots, the observed frequencies can differ considerably from the theoretical probabilities. As the number of shots increases, the frequencies approach $1/2$ for `00` and $1/2$ for `11`.


### **3 — Create the remaining Bell states**

Two qubits have four maximally entangled Bell states:

$$
\begin{aligned}
|\Phi^+\rangle &= \frac{|00\rangle+|11\rangle}{\sqrt{2}},\\
|\Phi^-\rangle &= \frac{|00\rangle-|11\rangle}{\sqrt{2}},\\
|\Psi^+\rangle &= \frac{|01\rangle+|10\rangle}{\sqrt{2}},\\
|\Psi^-\rangle &= \frac{|01\rangle-|10\rangle}{\sqrt{2}}.
\end{aligned}
$$

You have already created $|\Phi^+\rangle$. Now create and measure the remaining three.

**Tip:** besides the Hadamard and CNOT gates, you will need the $X$ and $Z$ gates.

Compare the measurement results for $|\Phi^+\rangle$ and $|\Phi^-\rangle$. Can measurement in the computational basis distinguish them? What happens for $|\Psi^+\rangle$ and $|\Psi^-\rangle$?


In [ ]:
# Create a circuit for the remaining Bell states

# Initialise the circuit
# num_qubits = 2
# qc_bell = qk.QuantumCircuit(num_qubits)

# First create |Phi+>, then apply the gate needed for the Bell state you want to prepare
# qc_bell.h(...)
# qc_bell.cx(..., ...)
# ...

# Draw the circuit
# qc_bell.draw(output="mpl")

# Add measurements and run the circuit
# qc_bell.measure_all()
# result = simulator.run(qc_bell, shots=1000).result()
# counts = result.get_counts()
# plot_histogram(counts)